<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2);">
  <tr>
    <td>
      <h1 style="margin-bottom: 0; color: black; font-size: clamp(1.5rem, 2.5vw, 2.5rem);">
        Drone RL with LLM-Guided Curriculum Learning
      </h1>
      <p style="margin-top: 8px; color: black; font-size: 1rem;">
        EVA Deep Reinforcement Learning &ndash; DRL Project, Spring 2026<br>
        Rino M. Albertin  
      </p>
    </td>
    <td align="right">
      <img src="docs/images/OST_Logo_DE_RGB@2000ppi.png" alt="OST Logo" width="250">
    </td>
  </tr>
</table>

## Introduction

<div style="display: flex; gap: 24px; align-items: flex-start; width: 100%;">

<div style="flex: 1.15; min-width: 0;">

This project investigates reinforcement learning for single-drone trajectory tracking in simulation. The goal is to train a drone policy that follows reference trajectories such as hover targets, straight lines, vertical movements and more complex paths.

The environment is based on PyBullet and `gym-pybullet-drones`. A Proximal Policy Optimization (PPO) agent is trained and evaluated on trajectory-tracking tasks. The main focus of the project is not only whether PPO can learn one fixed trajectory, but how different training strategies affect learning and generalization.

The project compares three training approaches:

1. direct PPO training,
2. a manually designed curriculum,
3. an LLM-guided curriculum.

The LLM is used only for curriculum generation, not for direct drone control.

</div>

<div style="flex: 0.85; min-width: 0;">
  <img src="docs/media/hard_scenario_llm_curriculum_directrpm_dynprev_m-taskdist_medium_seed0_scenario_rollout.gif"
       alt="Example drone trajectory tracking rollout"
       style="width: 100%; height: auto; display: block;">
</div>

</div>

## Objective and Approach

The objective is to compare how different training strategies influence the tracking performance of a single-drone reinforcement learning agent.

The central research question is:

> Can curriculum learning improve PPO-based drone trajectory tracking compared to direct training, and can a local LLM be used as a safe curriculum task proposer?

The project compares three training approaches. The first baseline is direct PPO training, where the agent is trained directly on the target task distribution. The second approach is a manually designed curriculum, where the agent first learns simpler stabilization and movement tasks before progressing to harder tracking tasks. The third approach is an LLM-guided curriculum. In this setup, the first stage is a deterministic hover bootstrap, while later stages are proposed by a local LLM.

The LLM is not used as a controller. It does not send actions to the drone and it does not generate executable simulation code. Its role is limited to proposing structured curriculum tasks. These proposals are only accepted if they pass deterministic validation.

The main comparison is based on the tracking performance after training. The most important evaluation aspects are tracking error, termination behaviour, failure classification and qualitative trajectory plots or GIFs.

## Environment and Agent

The agent is trained in a single-drone trajectory-tracking environment. The environment wraps a PyBullet drone simulation and exposes a compact Gymnasium-compatible reinforcement learning interface.

At each step, the agent observes the drone state and the current reference position. The policy then outputs an action that is applied to the simulated drone. The reward penalizes tracking error and action effort.

### State space S

The observation contains:

| Component | Meaning |
|---|---|
| Current position | Current drone position in x, y and z direction |
| Reference position | Desired reference position at the current trajectory step |
| Position error | Difference between current and reference position |
| Trajectory progress | Normalized progress through the trajectory |
| Dynamics observation | Velocity, attitude and angular velocity |
| Previous action | Previous PPO-facing action |

The dynamics observation and previous-action observation are enabled in the main training configurations. This gives the policy more information about the physical state of the drone and its recent control behaviour.

### Action space A

Two action interfaces are implemented:

| Interface | Description |
|---|---|
| `pid_position` | PPO outputs normalized target-position commands. These are mapped to PID target positions. |
| `direct_rpm` | PPO outputs normalized motor commands that are mapped to motor RPM values. |

The `pid_position` interface is the main learning interface because it provides a higher-level and more stable control problem. The `direct_rpm` interface is more direct and therefore harder, because the policy has to learn lower-level motor behaviour.

## Reward Design

The reward is designed for trajectory tracking. It penalizes the Euclidean distance between the current drone position and the reference position. In addition, it penalizes the squared magnitude of the applied action to discourage unnecessarily large control commands.

The implemented reward has the form:

$$
r_t = - w_p \cdot \lVert p_t - p_t^{ref} \rVert_2 - w_a \cdot \lVert a_t \rVert_2^2
$$

where:  
- $p_t$: Current drone position  
- $p_t^{ref}$: Reference position at the current trajectory step  
- $a_t$: Applied action passed to the environment  
- $w_p$: Position error weight  
- $w_a$: Action cost weight  

In the current implementation, the default weights are:

$$
w_p = 1.0, \qquad w_a = 0.01
$$

The reward is therefore highest when the drone stays close to the reference trajectory and uses small control actions. Large tracking errors and large actions both reduce the reward.

## Algorithm

The project uses Proximal Policy Optimization (PPO) from Stable-Baselines3.

PPO is a policy optimization method. Instead of learning an explicit action-value table or Q-function and selecting the action with the highest Q-value as in Q-learning approaches, PPO directly optimizes a parameterized policy. The policy maps the current observation to an action distribution from which actions are sampled during training. 

The implementation follows an actor-critic structure. The actor represents the policy and predicts the action distribution for the current observation. The critic estimates the value of the current state. This value estimate is used to compute advantages, which measure whether an action performed better or worse than expected. PPO updates the policy using a clipped objective. The clipping limits how much the new policy is allowed to deviate from the previous policy in one update step. This makes training more stable than an unconstrained policy-gradient update, which is important for continuous-control tasks such as drone trajectory tracking.

The model used for prediction is a multilayer perceptron policy (`MlpPolicy`). The default architecture uses two hidden layers with 128 neurons each for the actor and critic networks. During training, the policy predicts actions stochastically to allow exploration. During evaluation, actions are predicted deterministically to measure the learned tracking behaviour consistently.

PPO was selected because it is a stable and widely used baseline algorithm for continuous-control reinforcement learning tasks. It is well suited for this project because the drone action space is continuous and the goal is to learn a smooth control policy for trajectory tracking.

## Training Setup

Training is performed from configuration files, not manually inside the notebook. This keeps the experiments reproducible and separates long-running training from the final report.

The main PPO setup uses:

| Parameter | Value |
|---|---:|
| Algorithm | PPO |
| Policy | MlpPolicy |
| Learning rate α | 0.0003 |
| Discount factor γ | 0.99 |
| GAE λ | 0.95 |
| PPO clip range | 0.2 |
| Entropy coefficient | 0.001 |
| Number of vectorized environments | 4 |
| Training timesteps per main run | 500,000 |
| Evaluation steps | 240 |

The main experiments use vectorized environments to collect PPO rollouts more efficiently. Each environment is seeded deterministically.

## Experiments

The experiments are grouped into three main comparisons. First, selected PPO settings are varied to study their effect on learning. Second, the two action interfaces are compared. Third, the main training strategies are evaluated.

An important part of the experimental design is how training tasks are sampled. The direct PPO baselines are not trained on one fixed trajectory only. Instead, they use randomized task distributions. In the `tracking_medium` setup, a new task is sampled at the beginning of each episode and then kept fixed during that episode. This exposes the policy to a range of trajectory types instead of letting it memorize one reference path.

The `tracking_medium` distribution contains several task families:

- stabilization tasks, such as hover and takeoff stabilization,
- basic tracking tasks, such as straight lines and start-hold-then-line movements,
- piecewise-linear paths, such as polylines, L-shapes, rectangles and squares,
- curved paths, such as circles, ellipses and figure-eight trajectories,
- altitude-changing tasks, such as vertical up-down, angled vertical, delayed-altitude polylines and multi-height polylines.

This creates a training distribution that still focuses on basic trajectory tracking, but also exposes the policy to more varied trajectory shapes.

The curriculum approaches use a different structure. They do not immediately train on the full `tracking_medium` distribution. Instead, they progress through easier or more focused stages. Each stage still contains variation, for example in target position, movement length, direction, duration or height. This means that the curriculum stages are not simple repetitions of one identical trajectory, but controlled task distributions with a narrower difficulty range. In the manual curriculum, this progression is fixed in advance. The stages move from hover stabilization to vertical movements, short line movements, polylines and finally the broader medium tracking distribution. In the LLM curriculum, the first stage is a deterministic hover-bootstrap stage. Later stages are proposed by the local LLM, but only from structured and validated task definitions or known task distributions.

### 1. PPO Variant Comparison

A limited PPO variant comparison is included. The goal is not to perform a full hyperparameter sweep, but to compare selected settings that are relevant for training stability and tracking performance.

| Variant | Changed setting | Purpose |
|---|---|---|
| Default PPO | Baseline PPO configuration | Reference setting for all comparisons |
| `net256` | Larger policy and value network | Test whether a larger network improves tracking |
| `low_lr` | Lower learning rate $\alpha = 0.0001$ | Test whether slower policy updates improve stability |
| `gamma095` | Lower discount factor $\gamma = 0.95$ | Test whether a shorter effective planning horizon changes tracking performance |
| `ent005` | Higher entropy coefficient | Encourage more exploration |
| `clip010` | Smaller PPO clip range | Make policy updates more conservative |
| `targetkl015` | Lower target KL limit | Stop overly large policy updates earlier |

### 2. Action-Interface Comparison

The project compares `pid_position` and `direct_rpm`.

The `pid_position` interface is expected to be easier because the policy outputs target-position commands, while the lower-level PID controller handles part of the stabilization. This makes it the main interface for the report.

The `direct_rpm` interface is expected to be harder because the policy outputs motor-level commands. This gives the agent more direct control, but also makes the learning problem less stable.

### 3. Training Strategy Comparison

The main method comparison evaluates direct PPO, manual curriculum learning and LLM-guided curriculum learning.

Direct PPO is the baseline trained directly on the target task distribution. It therefore sees a broader task variety from the beginning of training.

The manual curriculum follows a fixed progression from easier to harder task distributions. This gives the agent a more structured learning path, but the progression itself is hand-designed.

The LLM curriculum also starts with a simple hover-bootstrap stage. After that, a local LLM proposes the next curriculum stages based on previous training feedback. These proposals are not used directly. They are accepted only if they pass deterministic validation before training.

## Evaluation Protocol

The trained policies are evaluated separately from training. Evaluation is based on deterministic rollout settings and fixed task definitions.

The evaluation answers three questions. Own-task evaluation tests how well the policy performs on its representative training task. Generalization evaluation tests how well the policy performs on fixed tasks outside the exact training episode. Scenario evaluation tests how the policy behaves in selected visual scenarios.

The most important quantitative metrics are:

- mean tracking error during the active tracking phase,
- mean position error over the full rollout,
- final position error,
- maximum position error,
- number of terminations,
- number of truncations,
- automatic failure-mode classification.

The failure classification can identify issues such as early termination, repeated truncation, action saturation, insufficient motion, overshoot, z-instability, attitude instability or no detected failure.

The evaluation does not rely on a single success-rate number. Instead, it combines tracking-error metrics, termination and truncation counts, and automatic failure-mode classification. Visual evaluation is also important because a low scalar error does not always fully describe the behaviour of the drone. Therefore, trajectory plots and GIFs are used.

## Results



In [3]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display as ipy_display

from src import evaluation

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

runs_root = evaluation.report.find_default_runs_root()
report_metric_rows = evaluation.report.build_report_metric_table(runs_root)
report_metrics_df = pd.DataFrame(report_metric_rows, columns=evaluation.report.REPORT_METRIC_OUTPUT_COLUMNS)
compact_columns = [column for column in evaluation.report.compact_report_columns() if column in report_metrics_df.columns]
report_metric_comparison = report_metrics_df.loc[:, compact_columns]
ipy_display(report_metric_comparison)
report_metric_comparison.to_csv("report_metric_comparison.csv", index=False)


,run_name,method,action_interface,variant,evaluation_name,suite_task_name,task_shape,mean_tracking_error_m,mean_position_error_m,final_position_error_m,max_position_error_m,mean_eval_reward,failure_status,primary_failure
0,curriculum_manual_directrpm_dynprev_m-taskdist...,Manual curriculum,direct_rpm,default,scenarios,easy,None,0.138559,0.138559,0.230404,0.340324,-0.144607,failed,truncated
1,curriculum_manual_directrpm_dynprev_m-taskdist...,Manual curriculum,direct_rpm,default,scenarios,hard,None,0.151839,0.151839,0.135182,0.311277,-0.156487,failed,truncated
2,curriculum_manual_directrpm_dynprev_m-taskdist...,Manual curriculum,direct_rpm,default,scenarios,medium,None,0.140998,0.140998,0.146672,0.322914,-0.147133,failed,truncated
3,curriculum_manual_directrpm_dynprev_m-taskdist...,Manual curriculum,direct_rpm,default,training,None,hover_stabilization,0.037347,0.065436,0.005891,0.289359,-0.069050,successful,no_failure_detected
4,curriculum_manual_directrpm_dynprev_m-taskdist...,Manual curriculum,direct_rpm,default,training,None,vertical,0.054529,0.070045,0.037170,0.288215,-0.076323,successful,no_failure_detected
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416,llm_curriculum_pid_dynprev_m-taskdist_medium_s...,LLM curriculum,pid_position,default,training,None,start_hold_then_short_line,0.071544,0.086413,0.291553,0.291553,-0.097755,successful,no_failure_detected
417,llm_curriculum_pid_dynprev_m-taskdist_medium_s...,LLM curriculum,pid_position,default,training,None,line,0.039452,0.056793,0.038431,0.283863,-0.064927,successful,no_failure_detected
418,llm_curriculum_pid_dynprev_m-taskdist_medium_s...,LLM curriculum,pid_position,default,training,None,ellipse,0.147987,0.148412,0.266286,0.283863,-0.161129,successful,no_failure_detected
419,llm_curriculum_pid_dynprev_m-taskdist_medium_s...,LLM curriculum,pid_position,default,training,None,start_hold_then_short_line,0.045715,0.063562,0.026586,0.283863,-0.075575,successful,no_failure_detected


## Discussion


## Conclusion and Outlook



## References

- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., and Klimov, O. “Proximal Policy Optimization Algorithms.” arXiv:1707.06347, 2017.
- Raffin, A., Hill, A., Gleave, A., Kanervisto, A., Ernestus, M., and Dormann, N. “Stable-Baselines3: Reliable Reinforcement Learning Implementations.” Journal of Machine Learning Research, 2021.
- Panerati, J., Zheng, H., Zhou, S., Xu, J., Prorok, A., and Schoellig, A. P. “Learning to Fly: A Gym Environment with PyBullet Physics for Reinforcement Learning of Multi-agent Quadcopter Control.” 2021.
- Coumans, E. and Bai, Y. PyBullet, a Python module for physics simulation.
- Farama Foundation. Gymnasium documentation and API.

ChatGPT by OpenAI was used for linguistic editing, report structuring and support with code snippets.  
The author remains fully responsible for the technical correctness, implementation and final content of this work.

## Declaration of Independence
I hereby certify that I have written this thesis independently and have not used any auxiliary materials other than those indicated. The passages of the work, which are taken in the wording or the sense after other works (to it also Internet sources count), were marked under indication of the source.

<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2); margin-top:20px;">
  <tr>
    <td align="left">
      <img src="docs/images/Unterschrift.png" alt="Unterschrift" style="height:150px;">
    </td>
  </tr>
</table>